In [ ]:
"""LAB 4 - Message Batches API.
 
Goal: submit many independent requests at 50% price, collect results by custom_id.
 
WHEN TO BATCH: large volumes, NOT latency-sensitive (nightly classification, bulk
summaries, evals, backfills). Results usually arrive within an hour, up to 24h, else expire.
WHEN NOT TO: interactive chat, anything a user waits on, or steps that depend on the previous answer.
 
Usage:
    python lab4_batch.py submit      # creates the batch, saves its id to batch_id.txt
    python lab4_batch.py collect     # run later; prints status, then results
"""


In [45]:
import pathlib
import sys

from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request

from common import MODEL, banner, get_client

client = get_client()
ID_FILE = pathlib.Path("batch_id.txt")

TICKETS = {
    "t-001":"Customer cannot log in after password reset.",
    "t-002":"Invoice shows  duplicate charge for September.",
    "t-003":"Feature request: export dashboards to PDF.",
    "t-004":"Server room temperature alaram is firing",
    "t-005":"How do I add a new user to my team?",
    "t-006":"Server room power cut-off"

}

def submit() -> None:
    requests = [
        Request(
            custom_id = tid,
            params = MessageCreateParamsNonStreaming(
                model=MODEL,
                max_tokens=60,
                system="Classify the ticket as one of :auth, billing, feature, infra, how-to" "Reply with Just label",
                messages = [{"role":"user","content":text}],
            ),
        )
         for tid,text in TICKETS.items()
    ]

    batch = client.messages.batches.create(requests=requests)
    ID_FILE.write_text(batch.id)
    print("Submitted:", batch.id, "| status:", batch.processing_status)
    print("Now: run 'python lab4_batch.py collect in a few minutes. ")

def collect() -> None:
    batch_id = ID_FILE.read_text().strip()
    batch = client.messages.batches.retrieve(batch_id)
    print("status:", batch.processing_status, "| counts", batch.request_counts)
    if batch.processing_status != "ended":
        print("Not Finished. Poll agian later (do not loop; every 30-60 s is plenty)")

        return

    banner("results (match on custom_id, NOT on position)")
    for item in client.messages.batches.results(batch_id):
        outcome = item.result
        if outcome.type == "succeeded":
            print(f"{item.custom_id}: {outcome.message.content[0].text.strip()}")
        elif outcome.type == "errored":
            kind = outcome.error.error.type
            # invalid_request_error -> FIX the request before resending
            # server-side errors   -> safe to resubmit as-is
            print(f"{item.custom_id}: ERRORED ({kind})")
        else:  # canceled / expired: not billed, resubmit if still needed
            print(f"{item.custom_id}: {outcome.type.upper()}")
 
    print("""
Check yourself:
- Which result types are billed? (succeeded only)
- Can batches be combined with prompt caching? (yes; discounts stack, but hits are best-effort)
- Why can't you use a batch for a chatbot reply?
    Latency will affect the response
- Stretch: add a 6th request with a deliberately invalid parameter. Is it rejected at submit time
  or reported as 'errored' in the results? Predict first, then observe.
    Submit time - No error, But Reported as Attribute error. Although process is completed. 
""")


if __name__ == "__main__":
    {"submit":submit,"collect":collect}.get(sys.argv[1] if len(sys.argv) >1 else "",lambda:print(__doc__))()

Automatically created module for IPython interactive environment


In [46]:
submit()

Submitted: msgbatch_01HYGnu1hxoNBS6Jszss5Epv | status: in_progress
Now: run 'python lab4_batch.py collect in a few minutes. 


In [47]:
collect()

status: in_progress | counts MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=6, succeeded=0)
Not Finished. Poll agian later (do not loop; every 30-60 s is plenty)


In [ ]:
batch_id = ID_FILE.read_text().strip()
client.messages.batches.results(batch_id)